In [1]:
from bh_molecule.instruments import Vis133M, load_wavecal_csv, csv_to_linear_formulas

In [2]:
FITS = './133mVis_169626.fits'
FITS_NEW = './193791.fits'
wavecal_csv = './133mVis_wavcal.csv'
# Load with formula mode
s = Vis133M(FITS, wavecal_csv, wavecal_mode="formula")
s_old = Vis133M(FITS, wavecal_csv, wavecal_mode="formula")
s_new = Vis133M(FITS_NEW, wavecal_csv, wavecal_mode="formula")

# Measure H-γ peak in old and new data
peak_old = s_old.measure_peak(0, pixel_window=(400, 600))
peak_new = s_new.measure_peak(0, pixel_window=(400, 600))

# Compute wavelength shift
delta = s.compute_wavelength_shift_from_peaks(peak_old[0], peak_new[0], channel=0)

# Apply correction
s.set_wavecal_shift(delta)

# Validate against original CSV
max_abs, rmse = s.compare_calibration_csv_vs_formula(0)

In [3]:
max_abs, rmse

(0.10161976329311528, 0.09936639821883017)

# Notes on this calibration check

In this notebook we:

- **Use the original CSV wavecal for all channels** via `Vis133M(..., wavecal_mode="formula")` to define the reference wavelength solution.
- **Load an old (binned) dataset and a new (unbinned) dataset** for the same spectrograph setup. The old data are effectively 1024 pixels in the dispersion direction, while the new data span 2048 pixels with no binning.
- **Measure the H-γ peak in both datasets** on channel 0 to determine how much the spectrum shifted in wavelength between the old and new configurations.
- **Compute a wavelength shift** from the difference in H-γ peak position (`compute_wavelength_shift_from_peaks`) and **apply that shift** to the new data's wavelength calibration (`set_wavecal_shift`).
- **Validate the adjusted calibration** by comparing the shifted formula-based solution against the original CSV (`compare_calibration_csv_vs_formula`).

The FITS headers for the new data do **not** correctly describe the change in binning, but based on the BH Q-branch lines and the H-γ position, the unbinned 2048-pixel wavelength solution appears consistent with the original binned (1024-pixel) calibration after applying this shift.